In [46]:
import argparse
import json
import csv
import glob

import os
import sys
sys.path.append("../..")
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np

import random

import nibabel as nib

#import seaborn as sns
import matplotlib.pyplot as plt

from multiprocessing import Pool
from functools import partial

import ipywidgets

from ipywidgets import interact, Text
import plotly.graph_objects as go

import anywidget

In [47]:
ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"
#ROOT_DIR = "/home/fehrdelt/bettik/"
SOOP_DIR = f"{ROOT_DIR}datasets/final_soop_dataset_small/"

In [48]:
adc_folder = os.path.join(SOOP_DIR, "adc_registered")
flair_folder = os.path.join(SOOP_DIR, "flair_registered")

masks_acute_folder = os.path.join(SOOP_DIR, "masks_acute_registered")
masks_chronic_folder = os.path.join(SOOP_DIR, "masks_chronic_registered")
masks_combined_folder = os.path.join(SOOP_DIR, "masks_combined_registered")

In [49]:
def scale_intensity_from_histogram_peak(input_image, target_value=1.0):
    # to be used only on mri images with intensities between 0 and 1

    #print(f"max input image value: {np.max(input_image)}")

    hist, bin_edges = np.histogram(input_image.flatten(), bins=100, range=(np.max(input_image)/15.0, np.max(input_image)))

    peak_value = bin_edges[np.argmax(hist)]

    normalized_image = input_image / peak_value * target_value

    return normalized_image

In [50]:
# Read CSV file with patient names to exclude
patients_to_exclude = pd.read_csv(f"{ROOT_DIR}AnoDiffExperiments/data_splits_lists/final_soop_dataset_small/exclude.csv", header=None)

In [51]:
# convert to list
patients_to_exclude = patients_to_exclude[0].tolist()

In [52]:
print(patients_to_exclude)

['sub-1041', 'sub-1091', 'sub-1112', 'sub-1119', 'sub-1133', 'sub-1138', 'sub-1166', 'sub-1175', 'sub-1183', 'sub-120', 'sub-1203', 'sub-1251', 'sub-1261', 'sub-1284', 'sub-1292', 'sub-1307', 'sub-1308', 'sub-1344', 'sub-1355', 'sub-148', 'sub-1491', 'sub-1496', 'sub-1502', 'sub-151', 'sub-1529', 'sub-1540', 'sub-1558', 'sub-1566', 'sub-1590', 'sub-1592', 'sub-1610', 'sub-1616', 'sub-1660', 'sub-1698', 'sub-171', 'sub-1717', 'sub-1727', 'sub-185', 'sub-188', 'sub-199', 'sub-208', 'sub-234', 'sub-249', 'sub-256', 'sub-279', 'sub-34', 'sub-343', 'sub-40', 'sub-438', 'sub-512', 'sub-52', 'sub-526', 'sub-560', 'sub-571', 'sub-586', 'sub-605', 'sub-617', 'sub-640', 'sub-7', 'sub-707', 'sub-764', 'sub-767', 'sub-785', 'sub-818', 'sub-846', 'sub-855', 'sub-887', 'sub-897', 'sub-949', 'sub-984', 'sub-989']


In [53]:
def analyze_patient(path_adc, path_flair, path_mask_acute, path_mask_chronic):

    # images
    adc_img = nib.load(path_adc)
    adc_data = scale_intensity_from_histogram_peak(adc_img.get_fdata())

    flair_img = nib.load(path_flair)
    flair_data = scale_intensity_from_histogram_peak(flair_img.get_fdata())

    # masks
    if path_mask_acute is not None:
        mask_acute_img = nib.load(path_mask_acute)
        mask_acute_data = mask_acute_img.get_fdata()
        # Get mean intensity for non-zero voxels of acute mask
        acute_mask_indices = mask_acute_data > 0
        
        mean_adc_acute = np.mean(adc_data[acute_mask_indices]) if np.any(acute_mask_indices) else np.nan
        mean_flair_acute = np.mean(flair_data[acute_mask_indices]) if np.any(acute_mask_indices) else np.nan
    else:
        mean_adc_acute = np.nan
        mean_flair_acute = np.nan

    if path_mask_chronic is not None:
        mask_chronic_img = nib.load(path_mask_chronic)
        mask_chronic_data = mask_chronic_img.get_fdata()
        chronic_mask_indices = mask_chronic_data > 0
        
        mean_adc_chronic = np.mean(adc_data[chronic_mask_indices]) if np.any(chronic_mask_indices) else np.nan
        mean_flair_chronic = np.mean(flair_data[chronic_mask_indices]) if np.any(chronic_mask_indices) else np.nan
    else:
        mean_adc_chronic = np.nan
        mean_flair_chronic = np.nan
    
    return mean_adc_acute, mean_flair_acute, mean_adc_chronic, mean_flair_chronic

In [54]:


def process_patient(file, adc_folder, flair_folder, masks_acute_folder, masks_chronic_folder, patients_to_exclude):
    patient_id = file.split(".")[0]
    if patient_id in patients_to_exclude:
        return None
    
    path_adc = os.path.join(adc_folder, file)
    path_flair = os.path.join(flair_folder, file)
    path_mask_acute = os.path.join(masks_acute_folder, file)
    path_mask_chronic = os.path.join(masks_chronic_folder, file)
    
    if not os.path.exists(path_flair):
        path_flair = None
    if not os.path.exists(path_mask_chronic):
        path_mask_chronic = None
    if not os.path.exists(path_mask_acute):
        path_mask_acute = None

    if path_mask_acute is None:
        return None

    mean_adc_acute, mean_flair_acute, mean_adc_chronic, mean_flair_chronic = analyze_patient(
        path_adc, path_flair, path_mask_acute, path_mask_chronic
    )

    return {
        "patient_id": patient_id,
        "mean_adc_acute": mean_adc_acute,
        "mean_flair_acute": mean_flair_acute,
        "mean_adc_chronic": mean_adc_chronic,
        "mean_flair_chronic": mean_flair_chronic
    }

files = os.listdir(adc_folder)
process_func = partial(process_patient, adc_folder=adc_folder, flair_folder=flair_folder, 
                       masks_acute_folder=masks_acute_folder, masks_chronic_folder=masks_chronic_folder,
                       patients_to_exclude=patients_to_exclude)

with Pool(processes=16) as pool:
    results = list(tqdm(pool.imap(process_func, files), total=len(files)))

results = [r for r in results if r is not None]
lesion_stats = pd.DataFrame(results)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1715/1715 [00:26<00:00, 64.40it/s]


In [55]:
lesion_stats.to_csv(f"{ROOT_DIR}AnoDiffExperiments/dataset_preprocessing/soop/lesion_stats.csv", index=False)

### **Dessiner la trajectoire théorique d'une lésion**

**Avec les heures/jours/semaines**

In [56]:
fig = go.Figure()

# Scatter plot for acute lesions
fig.add_trace(go.Scatter(
    x=lesion_stats['mean_flair_acute'],
    y=lesion_stats['mean_adc_acute'],
    mode='markers',
    name='Acute',
    marker=dict(color='cyan'),
    text=lesion_stats['patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Scatter plot for chronic lesions (excluding NaN values)
chronic_mask = lesion_stats['mean_adc_chronic'].notna()
fig.add_trace(go.Scatter(
    x=lesion_stats.loc[chronic_mask, 'mean_flair_chronic'],
    y=lesion_stats.loc[chronic_mask, 'mean_adc_chronic'],
    mode='markers',
    name='Chronic',
    marker=dict(color='red'),
    text=lesion_stats.loc[chronic_mask, 'patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Synthetic Healthy point
fig.add_trace(go.Scatter(
    x=[1.0],
    y=[1.0],
    mode='markers',
    name='Synthetic Healthy',
    marker=dict(color='black', symbol='x', size=10),
    hovertemplate='Synthetic Healthy<br>FLAIR: 1.0<br>ADC: 1.0<extra></extra>'
))

fig.update_layout(
    title='SOOP dataset mean FLAIR vs mean ADC intensities for Acute and Chronic Lesions',
    xaxis_title='Mean FLAIR lesion intensity (AU)',
    yaxis_title='Mean ADC lesion intensity (AU)',
    width=1200,
    height=900
)

fig.show()

### **Faire le graphique du dessus mais avec que les patients qui ont grosses lésions pour voir si ça sépare mieux**

**regarder patients Acute où la valeur d'ADC de la lésion est quand même à 1**

**regarder avant et après normalisation, si la histogram_peak_normalization a bien fonctionné**

In [57]:


fig = go.Figure()

# Scatter plot for acute lesions
fig.add_trace(go.Scatter(
    x=lesion_stats['mean_flair_acute'],
    y=lesion_stats['mean_adc_acute'],
    mode='markers',
    name='Acute',
    marker=dict(color='cyan'),
    text=lesion_stats['patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Scatter plot for chronic lesions (excluding NaN values)
chronic_mask = lesion_stats['mean_adc_chronic'].notna()
fig.add_trace(go.Scatter(
    x=lesion_stats.loc[chronic_mask, 'mean_flair_chronic'],
    y=lesion_stats.loc[chronic_mask, 'mean_adc_chronic'],
    mode='markers',
    name='Chronic',
    marker=dict(color='red'),
    text=lesion_stats.loc[chronic_mask, 'patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Synthetic Healthy point
fig.add_trace(go.Scatter(
    x=[1.0],
    y=[1.0],
    mode='markers',
    name='Synthetic Healthy',
    marker=dict(color='black', symbol='x', size=10),
    hovertemplate='Synthetic Healthy<br>FLAIR: 1.0<br>ADC: 1.0<extra></extra>'
))

fig.update_layout(
    title='SOOP dataset mean FLAIR vs mean ADC intensities for Acute and Chronic Lesions\nAdd "." at the end of patient ID to search and highlight',
    xaxis_title='Mean FLAIR lesion intensity (AU)',
    yaxis_title='Mean ADC lesion intensity (AU)',
    width=1200,
    height=900,

)

# Add a text input for search using annotations and shapes
patient_list = lesion_stats['patient_id'].tolist()



def search_patient(patient_name=''):
    if patient_name == '':
        fig.show()
        return
    
    if patient_name[-1] != '.': # Launch the search only if the last character is a dot
        fig.show()
        return
    else:
        patient_name = patient_name[:-1]  # Remove the dot for searching
    
    # Find matching patients
    
    matches = lesion_stats[lesion_stats['patient_id'].str.contains(patient_name, case=False)]
    
    if len(matches) == 0:
        print(f"No patient found matching '{patient_name}'")
        return
    
    # Highlight matching patients
    fig_search = go.Figure(fig)
    
    for _, row in matches.iterrows():
        # Add annotation for acute
        fig_search.add_annotation(
            x=row['mean_flair_acute'],
            y=row['mean_adc_acute'],
            text=row['patient_id'],
            showarrow=True,
            arrowhead=2,
            arrowcolor='black',
            font=dict(color='black', size=14)
        )
    
    fig_search.show()

interact(search_patient, patient_name=Text(value='', placeholder='Enter patient ID...', description='Search:'))

interactive(children=(Text(value='', description='Search:', placeholder='Enter patient ID...'), Output()), _do…

<function __main__.search_patient(patient_name='')>

**le search ça reveal que le acute**

In [58]:
fig = go.Figure()
fig.update_layout(
    xaxis=dict(range=[0, 2.5]),
    yaxis=dict(range=[0, 4.0])
)
# Scatter plot for acute lesions
fig.add_trace(go.Scatter(
    x=lesion_stats['mean_flair_acute'],
    y=lesion_stats['mean_adc_acute'],
    mode='markers',
    name='Acute',
    marker=dict(color='cyan'),
    text=lesion_stats['patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Scatter plot for chronic lesions (excluding NaN values)
chronic_mask = lesion_stats['mean_adc_chronic'].notna()
fig.add_trace(go.Scatter(
    x=lesion_stats.loc[chronic_mask, 'mean_flair_chronic'],
    y=lesion_stats.loc[chronic_mask, 'mean_adc_chronic'],
    mode='markers',
    name='Chronic',
    marker=dict(color='red'),
    text=lesion_stats.loc[chronic_mask, 'patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Synthetic Healthy point
fig.add_trace(go.Scatter(
    x=[1.0],
    y=[1.0],
    mode='markers',
    name='Synthetic Healthy',
    marker=dict(color='black', symbol='x', size=10),
    hovertemplate='Synthetic Healthy<br>FLAIR: 1.0<br>ADC: 1.0<extra></extra>'
))

def search_patient(patient_name=''):
    if patient_name == '':
        fig.show()
        return
    
    if patient_name[-1] != '.':
        fig.show()
        return
    else:
        patient_name = patient_name[:-1]
    
    matches = lesion_stats[lesion_stats['patient_id'].str.contains(patient_name, case=False)]
    
    if len(matches) == 0:
        print(f"No patient found matching '{patient_name}'")
        return
    
    # Create new figure with only matching patients
    fig_search = go.Figure()
    
    # Add only matching acute points
    fig_search.add_trace(go.Scatter(
        x=matches['mean_flair_acute'],
        y=matches['mean_adc_acute'],
        mode='markers',
        name='Acute',
        marker=dict(color='cyan', size=12),
        text=matches['patient_id'],
        hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
    ))
    
    # Add only matching chronic points
    matches_chronic = matches[matches['mean_adc_chronic'].notna()]
    if len(matches_chronic) > 0:
        fig_search.add_trace(go.Scatter(
            x=matches_chronic['mean_flair_chronic'],
            y=matches_chronic['mean_adc_chronic'],
            mode='markers',
            name='Chronic',
            marker=dict(color='red', size=12),
            text=matches_chronic['patient_id'],
            hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
        ))
    
    # Add annotations
    for _, row in matches.iterrows():
        fig_search.add_annotation(
            x=row['mean_flair_acute'],
            y=row['mean_adc_acute'],
            text=row['patient_id'],
            showarrow=True,
            arrowhead=2,
            arrowcolor='black',
            font=dict(color='black', size=14)
        )
    
    fig_search.update_layout(
        title=f'Matching patients: {patient_name}',
        xaxis_title='Mean FLAIR lesion intensity (AU)',
        yaxis_title='Mean ADC lesion intensity (AU)',
        width=1200,
        height=900,
        xaxis=dict(range=[0, 2.5]),
        yaxis=dict(range=[0, 4.0])
    )
    
    fig_search.show()

interact(search_patient, patient_name=Text(value='', placeholder='Enter patient ID...', description='Search:'))

interactive(children=(Text(value='', description='Search:', placeholder='Enter patient ID...'), Output()), _do…

<function __main__.search_patient(patient_name='')>

### Final version

In [59]:


fig_widget = go.FigureWidget()

# Scatter plot for acute lesions
fig_widget.add_trace(go.Scatter(
    x=lesion_stats['mean_flair_acute'],
    y=lesion_stats['mean_adc_acute'],
    mode='markers',
    name='Acute',
    marker=dict(color='cyan'),
    text=lesion_stats['patient_id'],
    customdata=lesion_stats['patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Scatter plot for chronic lesions (excluding NaN values)
chronic_mask = lesion_stats['mean_adc_chronic'].notna()
fig_widget.add_trace(go.Scatter(
    x=lesion_stats.loc[chronic_mask, 'mean_flair_chronic'],
    y=lesion_stats.loc[chronic_mask, 'mean_adc_chronic'],
    mode='markers',
    name='Chronic',
    marker=dict(color='red'),
    text=lesion_stats.loc[chronic_mask, 'patient_id'],
    customdata=lesion_stats.loc[chronic_mask, 'patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Synthetic Healthy point
fig_widget.add_trace(go.Scatter(
    x=[1.0],
    y=[1.0],
    mode='markers',
    name='Synthetic Healthy',
    marker=dict(color='black', symbol='x', size=10),
    hovertemplate='Synthetic Healthy<br>FLAIR: 1.0<br>ADC: 1.0<extra></extra>'
))

fig_widget.update_layout(
    title='SOOP dataset mean FLAIR vs mean ADC intensities for Acute and Chronic Lesions',
    xaxis_title='Mean FLAIR lesion intensity (AU)',
    yaxis_title='Mean ADC lesion intensity (AU)',
    width=1200,
    height=900
)

output_widget = ipywidgets.Output()

def on_click(trace, points, state):
    if points.point_inds:
        idx = points.point_inds[0]
        patient_id = trace.customdata[idx]
        
        with output_widget:
            output_widget.clear_output(wait=True)
            
            # Load and display images
            adc_path = os.path.join(adc_folder, f"{patient_id}.nii.gz")
            flair_path = os.path.join(flair_folder, f"{patient_id}.nii.gz")
            mask_acute_path = os.path.join(masks_acute_folder, f"{patient_id}.nii.gz")
            mask_chronic_path = os.path.join(masks_chronic_folder, f"{patient_id}.nii.gz")
            
            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            
            # ADC
            adc_img = nib.load(adc_path)
            adc_data = adc_img.get_fdata()
            mid_slice = adc_data.shape[2] // 2
            axes[0].imshow(adc_data[:, :, mid_slice].T, origin='lower', cmap='gray')
            axes[0].set_title(f'{patient_id} - ADC')
            axes[0].axis('off')
            
            # FLAIR
            if os.path.exists(flair_path):
                flair_img = nib.load(flair_path)
                flair_data = flair_img.get_fdata()
                axes[1].imshow(flair_data[:, :, mid_slice].T, origin='lower', cmap='gray')
                axes[1].set_title('FLAIR')
            axes[1].axis('off')
            
            # Acute mask
            if os.path.exists(mask_acute_path):
                mask_acute_img = nib.load(mask_acute_path)
                mask_acute_data = mask_acute_img.get_fdata()
                axes[2].imshow(adc_data[:, :, mid_slice].T, origin='lower', cmap='gray')
                axes[2].imshow(mask_acute_data[:, :, mid_slice].T, origin='lower', cmap='Reds', alpha=0.5)
                axes[2].set_title('Acute Mask')
            axes[2].axis('off')
            
            # Chronic mask
            if os.path.exists(mask_chronic_path):
                mask_chronic_img = nib.load(mask_chronic_path)
                mask_chronic_data = mask_chronic_img.get_fdata()
                axes[3].imshow(adc_data[:, :, mid_slice].T, origin='lower', cmap='gray')
                axes[3].imshow(mask_chronic_data[:, :, mid_slice].T, origin='lower', cmap='Blues', alpha=0.5)
                axes[3].set_title('Chronic Mask')
            axes[3].axis('off')
            
            plt.tight_layout()
            plt.show()

# Attach click handlers
fig_widget.data[0].on_click(on_click)
fig_widget.data[1].on_click(on_click)

display(fig_widget)
display(output_widget)

FigureWidget({
    'data': [{'customdata': array(['sub-1684', 'sub-1476', 'sub-326', ..., 'sub-870', 'sub-1122',
                                   'sub-225'], dtype=object),
              'hovertemplate': 'Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>',
              'marker': {'color': 'cyan'},
              'mode': 'markers',
              'name': 'Acute',
              'text': array(['sub-1684', 'sub-1476', 'sub-326', ..., 'sub-870', 'sub-1122',
                             'sub-225'], dtype=object),
              'type': 'scatter',
              'uid': '52f2d16e-1b8f-44d8-b129-3da8c5a6ee86',
              'x': {'bdata': ('IrmPno5z8T9bB0rEnfjxPy8jpIktBP' ... 'enYvQ/E5Xte54a8T8naFZDj4zzPw=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('g2+Yjbt35z8w+rD+ZQTpP69qrP5GMP' ... 'p6pfA/xYQYFbiS8j89MiiUj1XoPw=='),
                    'dtype': 'f8'}},
             {'customdata': array(['sub-1684', 'sub-1435', 'sub-882', 'sub-168', 'sub-1532', 'sub

Output()

In [23]:
fig_widget = go.FigureWidget()

# Scatter plot for acute lesions
fig_widget.add_trace(go.Scatter(
    x=lesion_stats['mean_flair_acute'],
    y=lesion_stats['mean_adc_acute'],
    mode='markers',
    name='Acute',
    marker=dict(color='cyan'),
    text=lesion_stats['patient_id'],
    customdata=lesion_stats['patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Scatter plot for chronic lesions (excluding NaN values)
chronic_mask = lesion_stats['mean_adc_chronic'].notna()
fig_widget.add_trace(go.Scatter(
    x=lesion_stats.loc[chronic_mask, 'mean_flair_chronic'],
    y=lesion_stats.loc[chronic_mask, 'mean_adc_chronic'],
    mode='markers',
    name='Chronic',
    marker=dict(color='red'),
    text=lesion_stats.loc[chronic_mask, 'patient_id'],
    customdata=lesion_stats.loc[chronic_mask, 'patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Synthetic Healthy point
fig_widget.add_trace(go.Scatter(
    x=[1.0],
    y=[1.0],
    mode='markers',
    name='Synthetic Healthy',
    marker=dict(color='black', symbol='x', size=10),
    hovertemplate='Synthetic Healthy<br>FLAIR: 1.0<br>ADC: 1.0<extra></extra>'
))

fig_widget.update_layout(
    title='SOOP dataset mean FLAIR vs mean ADC intensities for Acute and Chronic Lesions',
    xaxis_title='Mean FLAIR lesion intensity (AU)',
    yaxis_title='Mean ADC lesion intensity (AU)',
    width=1200,
    height=900
)

output_widget = ipywidgets.Output()

# Store selected point info
selected_info = {'trace_idx': None, 'point_idx': None}

def on_click(trace, points, state):
    if points.point_inds:
        idx = points.point_inds[0]
        patient_id = trace.customdata[idx]
        
        # Determine which trace was clicked
        trace_idx = list(fig_widget.data).index(trace)
        
        # Update marker sizes to highlight selected point
        for i, t in enumerate(fig_widget.data):
            if i == 2:  # Skip synthetic healthy point
                continue
            n_points = len(t.x)
            if i == trace_idx:
                # Highlight the selected point
                sizes = [8] * n_points
                sizes[idx] = 20
                fig_widget.data[i].marker.size = sizes
                # Add border to selected point
                line_widths = [0] * n_points
                line_widths[idx] = 3
                fig_widget.data[i].marker.line = dict(width=line_widths, color='black')
            else:
                # Reset other trace
                fig_widget.data[i].marker.size = 8
                fig_widget.data[i].marker.line = dict(width=0)
        
        with output_widget:
            output_widget.clear_output(wait=True)
            
            # Load and display images
            adc_path = os.path.join(adc_folder, f"{patient_id}.nii.gz")
            flair_path = os.path.join(flair_folder, f"{patient_id}.nii.gz")
            mask_acute_path = os.path.join(masks_acute_folder, f"{patient_id}.nii.gz")
            mask_chronic_path = os.path.join(masks_chronic_folder, f"{patient_id}.nii.gz")
            
            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            
            # ADC
            adc_img = nib.load(adc_path)
            adc_data = adc_img.get_fdata()
            mid_slice = adc_data.shape[2] // 2
            axes[0].imshow(adc_data[:, :, mid_slice].T, origin='lower', cmap='gray')
            axes[0].set_title(f'{patient_id} - ADC')
            axes[0].axis('off')
            
            # FLAIR
            if os.path.exists(flair_path):
                flair_img = nib.load(flair_path)
                flair_data = flair_img.get_fdata()
                axes[1].imshow(flair_data[:, :, mid_slice].T, origin='lower', cmap='gray')
                axes[1].set_title('FLAIR')
            axes[1].axis('off')
            
            # Acute mask
            if os.path.exists(mask_acute_path):
                mask_acute_img = nib.load(mask_acute_path)
                mask_acute_data = mask_acute_img.get_fdata()
                axes[2].imshow(adc_data[:, :, mid_slice].T, origin='lower', cmap='gray')
                axes[2].imshow(mask_acute_data[:, :, mid_slice].T, origin='lower', cmap='Reds', alpha=0.5)
                axes[2].set_title('Acute Mask')
            axes[2].axis('off')
            
            # Chronic mask
            if os.path.exists(mask_chronic_path):
                mask_chronic_img = nib.load(mask_chronic_path)
                mask_chronic_data = mask_chronic_img.get_fdata()
                axes[3].imshow(adc_data[:, :, mid_slice].T, origin='lower', cmap='gray')
                axes[3].imshow(mask_chronic_data[:, :, mid_slice].T, origin='lower', cmap='Blues', alpha=0.5)
                axes[3].set_title('Chronic Mask')
            axes[3].axis('off')
            
            plt.tight_layout()
            plt.show()

# Attach click handlers
fig_widget.data[0].on_click(on_click)
fig_widget.data[1].on_click(on_click)

display(fig_widget)
display(output_widget)

FigureWidget({
    'data': [{'customdata': array(['sub-1684', 'sub-1476', 'sub-326', ..., 'sub-870', 'sub-1122',
                                   'sub-225'], dtype=object),
              'hovertemplate': 'Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>',
              'marker': {'color': 'cyan'},
              'mode': 'markers',
              'name': 'Acute',
              'text': array(['sub-1684', 'sub-1476', 'sub-326', ..., 'sub-870', 'sub-1122',
                             'sub-225'], dtype=object),
              'type': 'scatter',
              'uid': 'a94d2941-e95f-4e99-b66c-b4dd7465a5be',
              'x': {'bdata': ('IrmPno5z8T9bB0rEnfjxPy8jpIktBP' ... 'enYvQ/E5Xte54a8T8naFZDj4zzPw=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('g2+Yjbt35z8w+rD+ZQTpP69qrP5GMP' ... 'p6pfA/xYQYFbiS8j89MiiUj1XoPw=='),
                    'dtype': 'f8'}},
             {'customdata': array(['sub-1684', 'sub-1435', 'sub-882', 'sub-168', 'sub-1532', 'sub

Output()

In [ ]:
fig_widget = go.FigureWidget()

# Scatter plot for acute lesions
fig_widget.add_trace(go.Scatter(
    x=lesion_stats['mean_flair_acute'],
    y=lesion_stats['mean_adc_acute'],
    mode='markers',
    name='Acute',
    marker=dict(color='cyan'),
    text=lesion_stats['patient_id'],
    customdata=lesion_stats['patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Scatter plot for chronic lesions (excluding NaN values)
chronic_mask = lesion_stats['mean_adc_chronic'].notna()
fig_widget.add_trace(go.Scatter(
    x=lesion_stats.loc[chronic_mask, 'mean_flair_chronic'],
    y=lesion_stats.loc[chronic_mask, 'mean_adc_chronic'],
    mode='markers',
    name='Chronic',
    marker=dict(color='red'),
    text=lesion_stats.loc[chronic_mask, 'patient_id'],
    customdata=lesion_stats.loc[chronic_mask, 'patient_id'],
    hovertemplate='Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>'
))

# Synthetic Healthy point
fig_widget.add_trace(go.Scatter(
    x=[1.0],
    y=[1.0],
    mode='markers',
    name='Synthetic Healthy',
    marker=dict(color='black', symbol='x', size=10),
    hovertemplate='Synthetic Healthy<br>FLAIR: 1.0<br>ADC: 1.0<extra></extra>'
))

fig_widget.update_layout(
    title='SOOP dataset mean FLAIR vs mean ADC intensities for Acute and Chronic Lesions',
    xaxis_title='Mean FLAIR lesion intensity (AU)',
    yaxis_title='Mean ADC lesion intensity (AU)',
    width=1200,
    height=900
)

output_widget = ipywidgets.Output()

# Store selected point info
selected_info = {'trace_idx': None, 'point_idx': None}

# Store loaded image data for slice scrolling
image_data_store = {'adc': None, 'flair': None, 'mask_acute': None, 'mask_chronic': None, 'patient_id': None}

# Slider widget for slice selection
slice_slider = ipywidgets.IntSlider(value=0, min=0, max=1, step=1, description='Slice:')

def update_slice_display(slice_idx):
    """Update the displayed images based on the selected slice."""
    if image_data_store['adc'] is None:
        return
    
    with output_widget:
        output_widget.clear_output(wait=True)
        
        adc_data = image_data_store['adc']
        flair_data = image_data_store['flair']
        mask_acute_data = image_data_store['mask_acute']
        mask_chronic_data = image_data_store['mask_chronic']
        patient_id = image_data_store['patient_id']
        
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        
        # ADC
        axes[0].imshow(adc_data[:, :, slice_idx].T, origin='lower', cmap='gray')
        axes[0].set_title(f'{patient_id} - ADC (slice {slice_idx})')
        axes[0].axis('off')
        
        # FLAIR
        if flair_data is not None:
            axes[1].imshow(flair_data[:, :, slice_idx].T, origin='lower', cmap='gray')
            axes[1].set_title(f'FLAIR (slice {slice_idx})')
        axes[1].axis('off')
        
        # Acute mask
        if mask_acute_data is not None:
            axes[2].imshow(adc_data[:, :, slice_idx].T, origin='lower', cmap='gray')
            axes[2].imshow(mask_acute_data[:, :, slice_idx].T, origin='lower', cmap='Reds', alpha=0.5)
            axes[2].set_title(f'Acute Mask (slice {slice_idx})')
        axes[2].axis('off')
        
        # Chronic mask
        if mask_chronic_data is not None:
            axes[3].imshow(adc_data[:, :, slice_idx].T, origin='lower', cmap='gray')
            axes[3].imshow(mask_chronic_data[:, :, slice_idx].T, origin='lower', cmap='Blues', alpha=0.5)
            axes[3].set_title(f'Chronic Mask (slice {slice_idx})')
        axes[3].axis('off')
        
        plt.tight_layout()
        plt.show()

def on_slider_change(change):
    """Handle slider value changes."""
    update_slice_display(change['new'])

slice_slider.observe(on_slider_change, names='value')

def on_click(trace, points, state):
    if points.point_inds:
        idx = points.point_inds[0]
        patient_id = trace.customdata[idx]
        
        # Determine which trace was clicked
        trace_idx = list(fig_widget.data).index(trace)
        
        # Update marker sizes to highlight selected point
        for i, t in enumerate(fig_widget.data):
            if i == 2:  # Skip synthetic healthy point
                continue
            n_points = len(t.x)
            if i == trace_idx:
                # Highlight the selected point
                sizes = [8] * n_points
                sizes[idx] = 20
                fig_widget.data[i].marker.size = sizes
                # Add border to selected point
                line_widths = [0] * n_points
                line_widths[idx] = 3
                fig_widget.data[i].marker.line = dict(width=line_widths, color='black')
            else:
                # Reset other trace
                fig_widget.data[i].marker.size = 8
                fig_widget.data[i].marker.line = dict(width=0)
        
        # Load images
        adc_path = os.path.join(adc_folder, f"{patient_id}.nii.gz")
        flair_path = os.path.join(flair_folder, f"{patient_id}.nii.gz")
        mask_acute_path = os.path.join(masks_acute_folder, f"{patient_id}.nii.gz")
        mask_chronic_path = os.path.join(masks_chronic_folder, f"{patient_id}.nii.gz")
        
        # Load ADC
        adc_img = nib.load(adc_path)
        image_data_store['adc'] = adc_img.get_fdata()
        image_data_store['patient_id'] = patient_id
        
        # Load FLAIR
        if os.path.exists(flair_path):
            flair_img = nib.load(flair_path)
            image_data_store['flair'] = flair_img.get_fdata()
        else:
            image_data_store['flair'] = None
        
        # Load Acute mask
        if os.path.exists(mask_acute_path):
            mask_acute_img = nib.load(mask_acute_path)
            image_data_store['mask_acute'] = mask_acute_img.get_fdata()
        else:
            image_data_store['mask_acute'] = None
        
        # Load Chronic mask
        if os.path.exists(mask_chronic_path):
            mask_chronic_img = nib.load(mask_chronic_path)
            image_data_store['mask_chronic'] = mask_chronic_img.get_fdata()
        else:
            image_data_store['mask_chronic'] = None
        
        # Update slider range
        num_slices = image_data_store['adc'].shape[2]
        slice_slider.max = num_slices - 1
        slice_slider.value = num_slices // 2  # Start at middle slice
        
        # Display the images
        update_slice_display(slice_slider.value)

# Attach click handlers
fig_widget.data[0].on_click(on_click)
fig_widget.data[1].on_click(on_click)

display(fig_widget)
display(slice_slider)
display(output_widget)

FigureWidget({
    'data': [{'customdata': array(['sub-1684', 'sub-1476', 'sub-326', ..., 'sub-870', 'sub-1122',
                                   'sub-225'], dtype=object),
              'hovertemplate': 'Patient: %{text}<br>FLAIR: %{x:.3f}<br>ADC: %{y:.3f}<extra></extra>',
              'marker': {'color': 'cyan'},
              'mode': 'markers',
              'name': 'Acute',
              'text': array(['sub-1684', 'sub-1476', 'sub-326', ..., 'sub-870', 'sub-1122',
                             'sub-225'], dtype=object),
              'type': 'scatter',
              'uid': '5599e6ea-b959-4b71-b8c5-bf82bb7e921f',
              'x': {'bdata': ('IrmPno5z8T9bB0rEnfjxPy8jpIktBP' ... 'enYvQ/E5Xte54a8T8naFZDj4zzPw=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('g2+Yjbt35z8w+rD+ZQTpP69qrP5GMP' ... 'p6pfA/xYQYFbiS8j89MiiUj1XoPw=='),
                    'dtype': 'f8'}},
             {'customdata': array(['sub-1684', 'sub-1435', 'sub-882', 'sub-168', 'sub-1532', 'sub

IntSlider(value=0, description='Slice:', max=1)

Output()